# -*- coding: utf-8 -*-
"""session-21-multiindex-objects.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/17l8EddlrS2Ed35frmeS6cHAvdf5Fbw-g
"""

In [ ]:
import numpy as np
import pandas as pd

### Series is 1D and DataFrames are 2D objects

- But why?
- And what exactly is index?

In [ ]:
# can we have multiple index? Let's try
index_val = [('cse',2019),('cse',2020),('cse',2021),('cse',2022),('ece',2019),('ece',2020),('ece',2021),('ece',2022)]
a = pd.Series([1,2,3,4,5,6,7,8],index=index_val)
a

In [ ]:
# The problem?
try:
    a['cse']
except KeyError as e:
    print("KeyError:", e)

### MultiIndex Series (Hierarchical Indexing)
Multiple index levels within a single index.

In [ ]:
# how to create multiindex object
# 1. pd.MultiIndex.from_tuples()
index_val = [('cse',2019),('cse',2020),('cse',2021),('cse',2022),('ece',2019),('ece',2020),('ece',2021),('ece',2022)]
multiindex = pd.MultiIndex.from_tuples(index_val)
multiindex.levels[1]

In [ ]:
# 2. pd.MultiIndex.from_product()
pd.MultiIndex.from_product([['cse','ece'],[2019,2020,2021,2022]])

#### Level inside multiindex object

In [ ]:
# creating a series with multiindex object
s = pd.Series([1,2,3,4,5,6,7,8],index=multiindex)
s

In [ ]:
# how to fetch items from such a series
s['cse']

#### Unstacking and Stacking

In [ ]:
temp = s.unstack()
temp

In [ ]:
temp.stack()

### MultiIndex DataFrame

In [ ]:
branch_df1 = pd.DataFrame(
    [
        [1,2],
        [3,4],
        [5,6],
        [7,8],
        [9,10],
        [11,12],
        [13,14],
        [15,16],
    ],
    index = multiindex,
    columns = ['avg_package','students']
)
branch_df1

In [ ]:
branch_df1['students']

#### MultiIndex Columns

In [ ]:
branch_df2 = pd.DataFrame(
    [
        [1,2,0,0],
        [3,4,0,0],
        [5,6,0,0],
        [7,8,0,0],
    ],
    index = [2019,2020,2021,2022],
    columns = pd.MultiIndex.from_product([['delhi','mumbai'],['avg_package','students']])
)
branch_df2

In [ ]:
branch_df2.loc[2019]

#### MultiIndex DataFrame in terms of both Index and Columns

In [ ]:
branch_df3 = pd.DataFrame(
    [
        [1,2,0,0],
        [3,4,0,0],
        [5,6,0,0],
        [7,8,0,0],
        [9,10,0,0],
        [11,12,0,0],
        [13,14,0,0],
        [15,16,0,0],
    ],
    index = multiindex,
    columns = pd.MultiIndex.from_product([['delhi','mumbai'],['avg_package','students']])
)
branch_df3

### Stacking and Unstacking

In [ ]:
branch_df3.stack().stack()

### Working with MultiIndex DataFrames

In [ ]:
# head and tail
branch_df3.head()

In [ ]:
# shape
branch_df3.shape

In [ ]:
# info
branch_df3.info()

In [ ]:
# duplicated -> isnull
branch_df3.duplicated()
branch_df3.isnull()

In [ ]:
# Extracting rows single
branch_df3.loc[('cse',2022)]

In [ ]:
# multiple
branch_df3.loc[('cse',2019):('ece',2020):2]

In [ ]:
# using iloc
branch_df3.iloc[0:5:2]

In [ ]:
# Extracting cols
branch_df3['delhi']['students']

In [ ]:
branch_df3.iloc[:,1:3]

In [ ]:
# Extracting both
branch_df3.iloc[[0,4],[1,2]]

In [ ]:
# sort index
# both -> descending -> diff order
# based on one level
branch_df3.sort_index(ascending=False)
branch_df3.sort_index(ascending=[False,True])
branch_df3.sort_index(level=0,ascending=[False])

In [ ]:
# transpose
branch_df3.transpose()

In [ ]:
# swaplevel
branch_df3.swaplevel(axis=1)

### Long Vs Wide Data

In [ ]:
# melt -> simple example branch
# wide to long
pd.DataFrame({'cse':[120]}).melt()

In [ ]:
pd.DataFrame({'cse':[120],'ece':[100],'mech':[50]}).melt(var_name='branch',value_name='num_students')

In [ ]:
pd.DataFrame(
    {
        'branch':['cse','ece','mech'],
        '2020':[100,150,60],
        '2021':[120,130,80],
        '2022':[150,140,70]
    }
).melt(id_vars=['branch'],var_name='year',value_name='students')

#### Real World Melt Example (COVID-19 Deaths and Confirmed)

In [ ]:
death = pd.read_csv('time_series_covid19_deaths_global.csv')
confirm = pd.read_csv('time_series_covid19_confirmed_global.csv')

death.head()

In [ ]:
confirm.head()

In [ ]:
death = death.melt(id_vars=['Province/State','Country/Region','Lat','Long'],var_name='date',value_name='num_deaths')
confirm = confirm.melt(id_vars=['Province/State','Country/Region','Lat','Long'],var_name='date',value_name='num_cases')

death.head()

In [ ]:
confirm.merge(death,on=['Province/State','Country/Region','Lat','Long','date'])[['Country/Region','date','num_cases','num_deaths']]

### Pivot Table

In [ ]:
import seaborn as sns
df = sns.load_dataset('tips')
df.head()

In [ ]:
df.groupby('sex')[['total_bill']].mean()

In [ ]:
df.groupby(['sex','smoker'])[['total_bill']].mean().unstack()

In [ ]:
df.pivot_table(index='sex',columns='smoker',values='total_bill')

In [ ]:
# aggfunc
df.pivot_table(index='sex',columns='smoker',values='total_bill',aggfunc='std')

In [ ]:
# all cols together
df.pivot_table(index='sex',columns='smoker')['size']

In [ ]:
# multidimensional
df.pivot_table(index=['sex','smoker'],columns=['day','time'],aggfunc={'size':'mean','tip':'max','total_bill':'sum'},margins=True)

In [ ]:
# margins
df.pivot_table(index='sex',columns='smoker',values='total_bill',aggfunc='sum',margins=True)

In [ ]:
# plotting graphs
df = pd.read_csv('expense_data.csv')
df.head()

In [ ]:
df['Category'].value_counts()

In [ ]:
df.info()

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df.info()

In [ ]:
df['month'] = df['Date'].dt.month_name()
df.head()

In [ ]:
df.pivot_table(index='month',columns='Category',values='INR',aggfunc='sum',fill_value=0).plot()

In [ ]:
df.pivot_table(index='month',columns='Income/Expense',values='INR',aggfunc='sum',fill_value=0).plot()

In [ ]:
df.pivot_table(index='month',columns='Account',values='INR',aggfunc='sum',fill_value=0).plot()